# Scalar Theory Playground

A tiny notebook for poking at the current `pychete` API. It builds a light scalar `phi^4` theory, prints the Lagrangian and EOM in LaTeX, then integrates out a heavy scalar at tree level.

Run this with the managed environment from a shell where the Symbolica license is available, for example:

```sh
source "$HOME/.bashrc"
source dependencies/.venv/bin/activate
```

If you launch Jupyter from that environment, the displayed equations below should render as LaTeX.

In [8]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src" / "pychete").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

try:
    from IPython.display import Markdown, Math, display
except ImportError:
    class Markdown(str):
        pass

    class Math(str):
        pass

    def display(value):
        print(value)

from symbolica import PrintMode

from pychete import (
    FieldMassKind,
    Theory,
    canonical_string,
    relabel_dummy_indices,
    s,
)

FORMAT_OPTIONS = {
    "max_line_length": None,
    "color_top_level_sum": False,
    "color_builtin_symbols": False,
    "bracket_level_colors": None,
    "print_ring": False,
    "multiplication_operator": "*",
    "num_exp_as_superscript": False,
}


def as_symbolica(expr):
    return expr.format(mode=PrintMode.Symbolica, **FORMAT_OPTIONS)


def as_latex(expr):
    return expr.format(mode=PrintMode.Latex, **FORMAT_OPTIONS)


def show_expr(label, expr, *, canonical=False):
    display(Markdown(f"**{label}**"))
    display(Math(as_latex(expr)))
    print(as_symbolica(expr))
    if canonical:
        print("\ncanonical:")
        print(canonical_string(expr))

## 1. A light scalar `phi^4` theory

This is the simplest useful scalar example: a real light scalar with mass `m` and quartic coupling `lambda`.

In [3]:
phi4 = Theory("phi4_playground")
phi = phi4.define_field(
    "phi",
    s.Scalar,
    self_conjugate=True,
    mass=(FieldMassKind.LIGHT, "m"),
)
lam = phi4.define_coupling("lambda", self_conjugate=True)

lagrangian_phi4 = phi4.free_lag(phi) - s.twenty_fourth * lam() * phi() ** 4

display(Markdown("Field and coupling handles render with their notebook hooks:"))
display(phi)
display(lam)

show_expr("phi^4 Lagrangian", lagrangian_phi4)

Field and coupling handles render with their notebook hooks:

**phi^4 Lagrangian**

<IPython.core.display.Math object>

-1/2*phi^2*m^2-1/24*phi^4*lambda+1/2*D[d0](phi)^2


`Theory.derive_eom(...)` gives the Euler-Lagrange equation for a field. The expression below is the left-hand side of the equation of motion.

In [4]:
eom_phi = phi4.derive_eom(lagrangian_phi4, phi)

show_expr("EOM for phi", eom_phi)
show_expr("Mass expression attached to phi", phi4.mass_expr(phi.definition))

**EOM for phi**

<IPython.core.display.Math object>

-phi*m^2-D[d, d](phi)-1/6*phi^3*lambda


**Mass expression attached to phi**

<IPython.core.display.Math object>

m


## 2. A heavy scalar matched onto a light theory

Now add a heavy real scalar `S` with mass `M` and interaction `-g S phi^2 / 2`. This is the small tree-level matching example covered by the tests.

In [5]:
heavy_scalar = Theory("heavy_scalar_playground")
S_field = heavy_scalar.define_field(
    "S",
    s.Scalar,
    self_conjugate=True,
    mass=(FieldMassKind.HEAVY, "M"),
)
light_phi = heavy_scalar.define_field("phi", s.Scalar, self_conjugate=True, mass=0)
g = heavy_scalar.define_coupling("g", self_conjugate=True)

lagrangian_heavy = (
    heavy_scalar.free_lag(S_field, light_phi)
    - s.half * g() * S_field() * light_phi() ** 2
)

show_expr("UV Lagrangian", lagrangian_heavy)

**UV Lagrangian**

<IPython.core.display.Math object>

-1/2*S*phi^2*g-1/2*S^2*M^2+1/2*D[d0](S)^2+1/2*D[d0](phi)^2


The heavy-field EOM is solved order by order in the EFT expansion. The first nonzero term is the familiar algebraic source divided by `M^2`; the higher-order term contains derivatives of the light field.

In [9]:
eom_S = heavy_scalar.derive_eom(lagrangian_heavy, S_field)
solution = heavy_scalar.solve_heavy_scalar_eoms(lagrangian_heavy, eft_order=6)["S"]

show_expr("EOM for S", eom_S)

for order in (1, 2, 3):
    show_expr(f"S solution, EFT order {order}", relabel_dummy_indices(solution.orders[order]))

**EOM for S**

<IPython.core.display.Math object>

-S*M^2-D[d, d](S)-1/2*phi^2*g


**S solution, EFT order 1**

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

0


**S solution, EFT order 3**

<IPython.core.display.Math object>

phi*D[d0, d0](phi)*g/M^4+D[d0](phi)^2*g/M^4


Finally substitute the heavy-field solution back into the theory to get the matched light-field Lagrangian through dimension six.

In [7]:
matched_lagrangian = heavy_scalar.match(lagrangian_heavy, eft_order=6)

show_expr(
    "Matched light-field Lagrangian through dimension six",
    matched_lagrangian,
    canonical=True,
)

**Matched light-field Lagrangian through dimension six**

<IPython.core.display.Math object>

1/2*phi^2*D[d0](phi)^2*g^2/M^4+1/8*phi^4*g^2/M^2+1/2*D[d0](phi)^2

canonical:
1/2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::List(),pychete::List())^2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::List(),pychete::List(pychete::Index(pychete::dummy_index(0),pychete::Lorentz)))^2*pychete::Coupling(heavy_scalar_playground::coupling_g,pychete::List(),0)^2/pychete::Coupling(heavy_scalar_playground::coupling_M,pychete::List(),0)^4+1/8*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::List(),pychete::List())^4*pychete::Coupling(heavy_scalar_playground::coupling_g,pychete::List(),0)^2/pychete::Coupling(heavy_scalar_playground::coupling_M,pychete::List(),0)^2+1/2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::List(),pychete::List(pychete::Index(pychete::dummy_index(0),pychete::Lorentz)))^2
